In [26]:
%pip install pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [27]:
import pandas as pd

train= pd.read_csv("dataset/train.csv", encoding = 'utf-8-sig')
test= pd.read_csv("dataset/test.csv", encoding = 'utf-8-sig')

train.head()

,ID,input,output
0,TRAIN_00000,별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈...,별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자...
1,TRAIN_00001,잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ,잠만 자고 갈 때 좋네요. 잠옷도 줌 ㅋ
2,TRAIN_00002,절테 간면 않 된는 굣 멥몫,절대 가면 안 되는 곳 메모
3,TRAIN_00003,야... 칵컥 좋꾜 부됴 뼝 뚫렷썹 신원햐쥠만 닮패 넴센 밌쪄벅림. 샥퀘 핥류만 묵...,아... 가격 좋고 뷰도 뻥 뚫려서 시원하지만 담배 냄새 미쳐버림. 싸게 하루만 묵...
4,TRAIN_00004,집윈 축쳐눌료 딴너왓눈뎁 카셩뷔 좋곱 칼쿰한네올. 쩌럼한뒈 뮬콰 욺료토 잊쿄 빻토 ...,지인 추천으로 다녀왔는데 가성비 좋고 깔끔하네요. 저렴한데 물과 음료도 있고 방도 ...


### Train/validation 분리

In [28]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train,
    test_size=0.2,
    random_state=42
)

### 단어 사전 만들기

##### train 데이터로만 사전 생성

In [29]:
match_dict = {}

# 단어단위로 사전 생성
# for input_text, output_text in zip(train_data['input'],train_data['output']):
#     input_words = input_text.split()
#     output_words = output_text.split()
#     for iw, ow in zip(input_words, output_words):
#         match_dict[iw] = ow

# 5.5. 오류분석에서 찾은 문제에 대한 해결방안 실험 - 음절 단위로 사전 만들어서 적용
for input_text, output_text in zip(train_data['input'], train_data['output']):
    for iw, ow in zip(input_text, output_text):
        match_dict[iw] = ow

### 복원 적용



In [30]:
def replace_words(input_text, match_dict):
    words = input_text.split()
    replaced_words = [match_dict.get(word, word) for word in words]
    return " ".join(replaced_words)

### validation 복원

In [31]:
val_pred = val_data['input'].apply(
    lambda x: replace_words(x, match_dict)
)

### Accuracy 계산
단어 - Validation Accuracy: 0.004882379050155349
음절 - Validation Accuracy: 0.0004438526409232135

In [32]:
accuracy = (
    val_pred == val_data['output']
).mean()

print("Validation Accuracy:", accuracy)



Validation Accuracy: 0.0004438526409232135


1. 수빈 - F1 Score 계산

단어 - Validation Char F1: 0.5684
음절 - Validation Char F1: 0.4191

In [24]:
from collections import Counter

def char_f1(pred, answer):
    pred_chars = Counter(pred)
    ans_chars = Counter(answer)

    # 공통 문자 수 구하기
    common = sum((pred_chars&ans_chars).values())

    if common == 0:
        return 0.0
    
    # precision = TP / TP + FP
    precision = common / sum(pred_chars.values())
    # Recall = TP / TP + FN
    recall = common / sum(ans_chars.values())

    #F1 = 2 * (Precision * Recall) / (Precision + Recall)
    return 2 * (precision *recall) / (precision + recall)


In [25]:
scores = [char_f1(pred, ans) for pred, ans in zip(val_pred, val_data['output'])]
print(f'Validation Char F1: {sum(scores)/len(scores):.4f}')

Validation Char F1: 0.4191


#### test 데이터 전체를 사전 기반으로 *복원*

In [10]:
converted_reviews = test['input'].apply(
    lambda x: replace_words(x, match_dict)
).tolist()

### Submission

In [11]:
submission = pd.read_csv('dataset/sample_submission.csv', encoding = 'utf-8-sig')

In [12]:
submission['output'] = converted_reviews

In [13]:
submission.to_csv('dataset/submission_dictionary.csv', index = False, encoding = 'utf-8-sig')

5_5. 성능개선 - 오류분석
-> 사전에 없는 단어는 전혀 복원 못 한다는 문제를 발견
=> 음절 단위로 단어사전을 만들면 정확도가 올라가지 않을까라고 추측

In [14]:
val_data['pred'] = val_pred
val_data['f1'] = [char_f1(p, a) for p, a in zip(val_pred, val_data['output'])]

worst_cases = val_data.sort_values('f1').head(20)

for _, row in worst_cases.iterrows():
    print(f"입력: {row['input']}")
    print(f"예측: {row['pred']}")
    print(f"정답: {row['output']}")
    print(f"F1: {row['f1']:.2f}")
    print()

입력: 윅꾸쳄뀨뛰쁠륨 섶빗수 쿰칙한넷욥.
예측: 윅꾸쳄뀨뛰쁠륨 섶빗수 쿰칙한넷욥.
정답: 이그제큐티브룸 서비스 끔찍하네요.
F1: 0.17

입력: 캑끝햐콤 셥퓟쓰갉 콕끕윌탔.
예측: 캑끝햐콤 셥퓟쓰갉 콕끕윌탔.
정답: 깨끗하고 서비스가 고급이다.
F1: 0.20

입력: 샴쟝뉨 췬졀옜 뎁졉팥눈 낍뷰닢닐타.
예측: 샴쟝뉨 췬졀옜 뎁졉팥눈 낍뷰닢닐타.
정답: 사장님 친절에 대접받는 기분입니다.
F1: 0.21

입력: 눔슥웽셔 봇콕 놂랏눼옮.
예측: 눔슥웽셔 봇콕 놂랏눼옮.
정답: 뉴스에서 보고 놀랐네요.
F1: 0.23

입력: 써율태벙언콰 갹칵윤 곶입곰 쩡켤합닉따.
예측: 써율태벙언콰 갹칵윤 곶입곰 쩡켤합닉따.
정답: 서울대병원과 가까운 곳이고 청결합니다.
F1: 0.24

입력: 게창 쪽귄눈 엿숟선헤쓺냐 침굼문 쪼항오
예측: 게창 쪽귄눈 엿숟선헤쓺냐 침굼문 쪼항오
정답: 개장 초기는 어수선했으나 지금은 좋아요
F1: 0.24

입력: 쉬썲있 께꿋햐곯 치컨뜰뤼 췬절할씹뉘타.
예측: 쉬썲있 께꿋햐곯 치컨뜰뤼 췬절할씹뉘타.
정답: 시설이 깨끗하고 직원들이 친절하십니다.
F1: 0.24

입력: 셩쑤엌 쿄압읽랐 윈취룰 깜얀한먼 걋썽핑갸 꿴챤앝섶욤. 쪼옹학쿄용.
예측: 셩쑤엌 쿄압읽랐 윈취룰 깜얀한먼 걋썽핑갸 꿴챤앝섶욤. 쪼옹학쿄용.
정답: 성수역 코앞이라 위치를 감안하면 가성비가 괜찮았어요. 조용하고요.
F1: 0.25

입력: 캐쿳했곶 찐졀핫셧엇옮.
예측: 캐쿳했곶 찐졀핫셧엇옮.
정답: 깨끗했고 친절하셨어요.
F1: 0.25

입력: 캐윈적굵록 많쪼캅닙탸.
예측: 캐윈적굵록 많쪼캅닙탸.
정답: 개인적으로 만족합니다.
F1: 0.25

입력: 유넝차뷴들독 찐쩔햐쉬꼬 깩꿋한 싣썰립뉘타.
예측: 유넝차뷴들독 찐쩔햐쉬꼬 깩꿋한 싣썰립뉘타.
정답: 운영자분들도 친절하시고 깨끗한 시설입니다.
F1: 0.26

입력: 쟘잘린캅 펀학꽃 좋얏씁닉타.
예측: 쟘잘린캅 펀학꽃 좋얏씁닉타.
정답: 잠자리가 편하고 좋았습니다.
F1: 0.27

입력: 푤렴읊료 빵문헷써욜. 돗쉴략 맏잇